In [1]:
import torch

# ---- Model Definition ----
class AttentionGraph(torch.nn.Module):
    def __init__(self, B=1, M=8192, H=96, E=128):
        super().__init__()

        D = H * E
        F = E
        G = H * E
        C = 4 * H * E
        J = H * E

        # Register weights as parameters
        self.WV = torch.nn.Parameter(torch.randn(H, E, D))
        self.WK = torch.nn.Parameter(torch.randn(H, E, D))
        self.WQ = torch.nn.Parameter(torch.randn(H, E, D))
        self.WZ = torch.nn.Parameter(torch.randn(H, F, G))
        self.WFFA = torch.nn.Parameter(torch.randn(G, C))
        self.WFFB = torch.nn.Parameter(torch.randn(C, J))

    def forward(self, I_in: torch.Tensor):
        # I copy
        I = I_in

        # Projections
        V = torch.einsum("bmd,hed->bmhe", I, self.WV)
        K = torch.einsum("bmd,hed->bmhe", I, self.WK)
        Q = torch.einsum("bmd,hed->bmhe", I, self.WQ)

        # QK (note: p dimension = m dimension reused)
        QK = torch.einsum("bmhe,bphe->bmph", Q, K)

        # Softmax over p dimension
        QK_softmax = torch.softmax(QK, dim=2)

        # Attention value
        AV = torch.einsum("bmph,bphe->bmhe", QK_softmax, V)

        # Output projection
        Z = torch.einsum("bmhf,hfg->bmg", AV, self.WZ)

        # Feedforward
        FFA = torch.einsum("bmg,gc->bmc", Z, self.WFFA)
        FFB = torch.einsum("bmc,cj->bmj", FFA, self.WFFB)

        return FFB

In [2]:
model = AttentionGraph(B=1, M=128, H=8, E=32)

scripted_model = torch.jit.script(model)

B, M, H, E = 1, 128, 8, 32
D = H * E

I_in = torch.randn(B, M, D)

output = scripted_model(I_in)
print(output.shape)

torch.Size([1, 128, 256])


In [3]:
print(scripted_model.inlined_graph)

graph(%self : __torch__.AttentionGraph,
      %I_in.1 : Tensor):
  %2 : str = prim::Constant[value="bmc,cj->bmj"]() # /tmp/ipykernel_20746/2155914073.py:45:27
  %3 : str = prim::Constant[value="bmg,gc->bmc"]() # /tmp/ipykernel_20746/2155914073.py:44:27
  %4 : str = prim::Constant[value="bmhf,hfg->bmg"]() # /tmp/ipykernel_20746/2155914073.py:41:25
  %5 : str = prim::Constant[value="bmph,bphe->bmhe"]() # /tmp/ipykernel_20746/2155914073.py:38:26
  %6 : str = prim::Constant[value="bmhe,bphe->bmph"]() # /tmp/ipykernel_20746/2155914073.py:32:26
  %7 : NoneType = prim::Constant()
  %8 : str = prim::Constant[value="bmd,hed->bmhe"]() # /tmp/ipykernel_20746/2155914073.py:27:25
  %9 : int = prim::Constant[value=2]() # /tmp/ipykernel_20746/2155914073.py:35:43
  %WV : Tensor = prim::GetAttr[name="WV"](%self)
  %11 : Tensor[] = prim::ListConstruct(%I_in.1, %WV)
  %V.1 : Tensor = aten::einsum(%8, %11, %7) # /tmp/ipykernel_20746/2155914073.py:27:12
  %WK : Tensor = prim::GetAttr[name="WK"](%self)
  %1

In [4]:
with torch.profiler.profile(record_shapes=True) as prof:
    scripted_model(I_in)

print(prof.key_averages().table(sort_by="cpu_time_total"))

------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                 forward         0.87%     119.137us       100.00%      13.755ms      13.755ms             1  
            aten::einsum        10.22%       1.406ms        76.24%      10.487ms       1.311ms             8  
               aten::bmm        46.10%       6.341ms        49.07%       6.750ms     843.793us             8  
           aten::softmax         4.22%     580.672us        22.90%       3.149ms       3.149ms             1  
          aten::_softmax         6.81%     936.452us        18.67%       2.569ms       2.569ms             1  
             aten::clone         4.02%     553.123us        13.10%       1.802ms     163.781us            11  
 

/mnt/c/Users/lyoko/Desktop/College/Grad/6.5931/accelforge_env/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
ERROR:2026-04-13 10:27:53 20746:20746 DeviceProperties.cpp:48] gpuGetDeviceCount failed with code 35


In [5]:
"""
Short answer: no, there’s essentially no fusion or explicit parallel
scheduling happening here—this graph is a straight-line, eager-style
execution graph where each einsum is its own op.
"""

'\nShort answer: no, there’s essentially no fusion or explicit parallel\nscheduling happening here—this graph is a straight-line, eager-style\nexecution graph where each einsum is its own op.\n'

In [6]:
class AttentionGraph(torch.nn.Module):
    def __init__(self, B=1, M=128, H=8, E=32):  # keep smaller for testing
        super().__init__()

        D = H * E
        F = E
        G = H * E
        C = 4 * H * E
        J = H * E

        self.WV = torch.nn.Parameter(torch.randn(H, E, D))
        self.WK = torch.nn.Parameter(torch.randn(H, E, D))
        self.WQ = torch.nn.Parameter(torch.randn(H, E, D))
        self.WZ = torch.nn.Parameter(torch.randn(H, E, G))
        self.WFFA = torch.nn.Parameter(torch.randn(G, C))
        self.WFFB = torch.nn.Parameter(torch.randn(C, J))

    def forward(self, I_in: torch.Tensor):
        I = I_in

        # ---- PARALLEL SECTION ----
        fut_V = torch.jit.fork(torch.einsum, "bmd,hed->bmhe", I, self.WV)
        fut_K = torch.jit.fork(torch.einsum, "bmd,hed->bmhe", I, self.WK)
        fut_Q = torch.jit.fork(torch.einsum, "bmd,hed->bmhe", I, self.WQ)

        V = torch.jit.wait(fut_V)
        K = torch.jit.wait(fut_K)
        Q = torch.jit.wait(fut_Q)
        # ---- END PARALLEL ----

        QK = torch.einsum("bmhe,bphe->bmph", Q, K)
        QK_softmax = torch.softmax(QK, dim=2)

        AV = torch.einsum("bmph,bphe->bmhe", QK_softmax, V)

        Z = torch.einsum("bmhe,heg->bmg", AV, self.WZ)

        FFA = torch.einsum("bmg,gc->bmc", Z, self.WFFA)
        FFB = torch.einsum("bmc,cj->bmj", FFA, self.WFFB)

        return FFB

In [7]:
model = AttentionGraph(B=1, M=128, H=8, E=32)

scripted_model = torch.jit.script(model)

B, M, H, E = 1, 128, 8, 32
D = H * E

I_in = torch.randn(B, M, D)

output = scripted_model(I_in)
print(output.shape)

torch.Size([1, 128, 256])


In [8]:
print(scripted_model.inlined_graph)

graph(%self : __torch__.___torch_mangle_0.AttentionGraph,
      %I_in.1 : Tensor):
  %2 : str = prim::Constant[value="bmc,cj->bmj"]() # /tmp/ipykernel_20746/3409957454.py:39:27
  %3 : str = prim::Constant[value="bmg,gc->bmc"]() # /tmp/ipykernel_20746/3409957454.py:38:27
  %4 : str = prim::Constant[value="bmhe,heg->bmg"]() # /tmp/ipykernel_20746/3409957454.py:36:25
  %5 : str = prim::Constant[value="bmph,bphe->bmhe"]() # /tmp/ipykernel_20746/3409957454.py:34:26
  %6 : NoneType = prim::Constant()
  %7 : str = prim::Constant[value="bmhe,bphe->bmph"]() # /tmp/ipykernel_20746/3409957454.py:31:26
  %8 : str = prim::Constant[value="bmd,hed->bmhe"]() # /tmp/ipykernel_20746/3409957454.py:22:45
  %9 : int = prim::Constant[value=2]() # /tmp/ipykernel_20746/3409957454.py:32:43
  %WV : Tensor = prim::GetAttr[name="WV"](%self)
  %fut_V.1 : Future[Tensor] = prim::fork_0(%I_in.1, %WV, %8) # /tmp/ipykernel_20746/3409957454.py:22:16
  %WK : Tensor = prim::GetAttr[name="WK"](%self)
  %fut_K.1 : Future[Te

In [10]:
with torch.profiler.profile(record_shapes=True) as prof:
    scripted_model(I_in)

print(prof.key_averages().table())

------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                 forward         0.00%       0.000us             0      24.744ms      24.744ms             1  
       <forked function>         0.63%     213.014us        56.11%      19.015ms       6.338ms             3  
            aten::einsum         2.90%     982.010us        92.98%      31.510ms       3.939ms             8  
         aten::unsqueeze         1.18%     398.766us         1.34%     454.443us      18.178us            25  
        aten::as_strided         0.40%     135.981us         0.40%     135.981us       1.528us            89  
           aten::permute         0.74%     251.368us         0.89%     299.901us       7.498us            40  
 